In [2]:
import pandas as pd
BALL_PATH = "C:/Desktop/crickanal/data_processed/ball_by_ball_canonical.csv"
MATCH_PATH = "C:/Desktop/crickanal/data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"

df_ball = pd.read_csv(BALL_PATH, parse_dates=["date"])
df_match = pd.read_csv(MATCH_PATH, parse_dates=["date"])


In [3]:
df_model = df_ball[
    (df_ball["is_second_innings"] == 0) &   # FIRST INNINGS
    (df_ball["has_elo"] == True)
].copy()


In [4]:
df_model = df_model.sort_values(
    ["match_id", "innings", "over", "ball"]
)

df_model["future_runs"] = (
    df_model
    .groupby(["match_id", "innings"])["total_runs"]
    .transform(lambda x: x[::-1].cumsum()[::-1])
)


In [5]:
df_model = df_model.sort_values(
    ["match_id", "innings", "over", "ball"]
).copy()

df_model["future_runs"] = (
    df_model
    .groupby(["match_id", "innings"])["total_runs"]
    .transform(lambda x: x.iloc[::-1].cumsum().iloc[::-1])
)


In [6]:
df_model = df_model[
    (df_model["is_second_innings"] == 0) &
    (df_model["has_elo"] == True) &
    (df_model["balls_remaining"] > 0)
].copy()


In [7]:
df_model.groupby("balls_remaining")["future_runs"].mean().head()


balls_remaining
1     3.772103
2     5.628332
3     7.424275
4     9.236644
5    11.025053
Name: future_runs, dtype: float64

In [8]:
df_model.columns


Index(['match_id', 'innings', 'over', 'ball', 'batting_team', 'bowling_team',
       'batter', 'bowler', 'fielder', 'runs_off_bat', 'extras', 'total_runs',
       'is_wicket', 'runs_remaining', 'balls_remaining', 'wickets_lost',
       'current_run_rate', 'required_run_rate', 'req_per_ball', 'rr_diff',
       'is_second_innings', 'phase_powerplay', 'phase_death', 'date', 'league',
       'team1', 'team2', 'venue', 'city', 'latitude', 'longitude', 'tavg',
       'tmax', 'tmin', 'rhum', 'prcp', 'wspd', 'pres', 'tsun', 'batting_elo',
       'batting_team_matches', 'bowling_elo', 'bowling_team_matches',
       'elo_diff', 'bat_first_team', 'bat_first_flag', 'has_elo',
       'future_runs'],
      dtype='object')

In [9]:
df_model["runs_scored_so_far"] = (
    df_model
    .groupby(["match_id", "innings"])["total_runs"]
    .cumsum()
)


In [10]:
RPA_FEATURES = [
    "balls_remaining",
    "wickets_lost",
    "current_run_rate",
    "runs_scored_so_far",
    "phase_powerplay",
    "phase_death",
    "venue",
    "league",
    "batting_elo",
    "bowling_elo",
    "elo_diff"
]


In [11]:
TARGET = "future_runs"


In [12]:
df_model[["runs_scored_so_far", "balls_remaining"]].head(20)


,runs_scored_so_far,balls_remaining
10800,3,119
10801,3,118
10802,4,117
10803,4,116
10804,4,115
10805,4,114
10806,5,113
10807,7,112
10808,7,111
10809,7,110


In [13]:
match_dates = (
    df_model[["match_id", "date"]]
    .drop_duplicates()
    .sort_values("date")
)

train_cutoff = match_dates["date"].quantile(0.65)
val_cutoff   = match_dates["date"].quantile(0.85)


In [14]:
train = df_model[df_model["date"] <= train_cutoff].copy()

val = df_model[
    (df_model["date"] > train_cutoff) &
    (df_model["date"] <= val_cutoff)
].copy()

test = df_model[df_model["date"] > val_cutoff].copy()


In [15]:
train["match_id"].nunique(), val["match_id"].nunique(), test["match_id"].nunique()


(3799, 1166, 877)

In [16]:
X_train = train[RPA_FEATURES]
y_train = train[TARGET]

X_val = val[RPA_FEATURES]
y_val = val[TARGET]

X_test = test[RPA_FEATURES]
y_test = test[TARGET]


In [17]:
cat_cols = ["venue", "league"]
num_cols = [c for c in RPA_FEATURES if c not in cat_cols]

In [18]:
from category_encoders import CatBoostEncoder

cbe = CatBoostEncoder(cols=cat_cols, random_state=42)

X_train_enc = cbe.fit_transform(X_train, y_train)
X_val_enc   = cbe.transform(X_val)
X_test_enc  = cbe.transform(X_test)


In [19]:
X_train_enc.head()
X_train_enc.isna().sum().sum()


np.int64(229)

In [20]:
na_counts = X_train_enc.isna().sum()
na_counts[na_counts > 0].sort_values(ascending=False)


current_run_rate    229
dtype: int64

In [21]:
X_train_enc["current_run_rate"] = X_train_enc["current_run_rate"].fillna(0)
X_val_enc["current_run_rate"]   = X_val_enc["current_run_rate"].fillna(0)
X_test_enc["current_run_rate"]  = X_test_enc["current_run_rate"].fillna(0)


In [22]:
X_train_enc.isna().sum().sum()


np.int64(0)

In [23]:
X_train_enc.shape, y_train.shape

((459728, 11), (459728,))

In [24]:
X_val_enc.shape, y_val.shape


((141373, 11), (141373,))

In [25]:
X_test_enc.shape, y_test.shape


((106202, 11), (106202,))

In [35]:
X_train_enc.describe().T[["min", "max"]]


,min,max
balls_remaining,1.000000,120.000000
wickets_lost,0.000000,10.000000
current_run_rate,0.000000,66.000000
runs_scored_so_far,0.000000,272.000000
phase_powerplay,0.000000,1.000000
phase_death,0.000000,1.000000
venue,23.445749,238.601402
league,52.722099,198.701302
batting_elo,1321.405439,1594.876790
bowling_elo,1331.797876,1582.688453


In [26]:
y_train.describe()


count    459728.000000
mean         86.818225
std          50.215004
min           0.000000
25%          46.000000
50%          85.000000
75%         124.000000
max         273.000000
Name: future_runs, dtype: float64

In [27]:
from sklearn.ensemble import HistGradientBoostingRegressor

rpa_model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    min_samples_leaf=50,
    l2_regularization=0.1,
    random_state=42
)

rpa_model.fit(X_train_enc, y_train)


,loss,'squared_error'
,quantile,None
,learning_rate,0.05
,max_iter,300
,max_leaf_nodes,31
,max_depth,6
,min_samples_leaf,50
,l2_regularization,0.1
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


In [28]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Predict on train and test
y_train_pred = rpa_model.predict(X_train_enc)
y_test_pred  = rpa_model.predict(X_test_enc)

# --- Core regression metrics ---

# RMSE
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))

# MAE
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test  = mean_absolute_error(y_test, y_test_pred)

# R²
r2_train = r2_score(y_train, y_train_pred)
r2_test  = r2_score(y_test, y_test_pred)

print("RPA model performance:")
print(f"  Train RMSE: {rmse_train:.4f}")
print(f"  Test  RMSE: {rmse_test:.4f}")
print(f"  Train MAE : {mae_train:.4f}")
print(f"  Test  MAE : {mae_test:.4f}")
print(f"  Train R²  : {r2_train:.4f}")
print(f"  Test  R²  : {r2_test:.4f}")

# --- Optional: simple calibration / residual check in bins ---

def binned_residuals(y_true, y_pred, n_bins=10):
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
    df["bin"] = pd.qcut(df["y_pred"], q=n_bins, duplicates="drop")
    grouped = df.groupby("bin").agg(
        pred_mean=("y_pred", "mean"),
        true_mean=("y_true", "mean"),
        count=("y_true", "size")
    )
    grouped["residual_mean"] = grouped["true_mean"] - grouped["pred_mean"]
    return grouped

print("\nBinned residuals (test set):")
print(binned_residuals(y_test, y_test_pred, n_bins=10))


RPA model performance:
  Train RMSE: 17.2083
  Test  RMSE: 25.1234
  Train MAE : 12.6180
  Test  MAE : 18.4973
  Train R²  : 0.8826
  Test  R²  : 0.7878

Binned residuals (test set):
                     pred_mean   true_mean  count  residual_mean
bin                                                             
(-28.289, 20.796]    11.254556   14.328312  10621       3.073756
(20.796, 37.729]     29.335634   33.791337  10620       4.455703
(37.729, 53.208]     45.396360   51.482674  10620       6.086314
(53.208, 68.654]     61.116701   67.870433  10620       6.753732
(68.654, 83.846]     76.165704   84.362618  10620       8.196914
(83.846, 98.894]     91.395722  101.192750  10620       9.797027
(98.894, 114.447]   106.540765  116.254237  10620       9.713473
(114.447, 131.31]   122.830839  133.177589  10620      10.346750
(131.31, 148.783]   140.002178  150.396139  10620      10.393961
(148.783, 210.339]  160.779227  171.627719  10621      10.848491


C:\Users\kathe\AppData\Local\Temp\ipykernel_23128\67372744.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby("bin").agg(


In [38]:
val["pred_future_runs"] = rpa_model.predict(X_val_enc)


In [40]:
val.groupby("balls_remaining")["pred_future_runs"].mean().head(10)
# val.groupby("balls_remaining")["pred_future_runs"].mean().tail(10)


balls_remaining
1      3.748353
2      4.660765
3      6.184448
4      7.917055
5      9.529357
6     11.078715
7     12.840517
8     14.562465
9     16.223586
10    17.784354
Name: pred_future_runs, dtype: float64

In [44]:
val.groupby("balls_remaining")["pred_future_runs"].mean().tail(10)


balls_remaining
111    152.680957
112    153.730178
113    155.169603
114    156.235364
115    157.361100
116    158.505193
117    159.547830
118    160.386794
119    161.451546
120    161.911091
Name: pred_future_runs, dtype: float64

In [ ]:
val.groupby("phase_death")["pred_future_runs"].mean()


phase_death
0    104.537121
1     26.179682
Name: pred_future_runs, dtype: float64

In [42]:
val.groupby("phase_powerplay")["pred_future_runs"].mean()


phase_powerplay
0     61.814643
1    139.199831
Name: pred_future_runs, dtype: float64

In [43]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_val, val["pred_future_runs"])


17.86922776330624

In [49]:
val["exp_runs_per_ball"] = (
    val["pred_future_runs"] - 
    val.groupby(["match_id", "innings"])["pred_future_runs"].shift(-1)
)

val.groupby("phase_powerplay")["exp_runs_per_ball"].mean()
# val.groupby("phase_death")["exp_runs_per_ball"].mean()


phase_powerplay
0    1.275035
1    1.329954
Name: exp_runs_per_ball, dtype: float64

In [50]:
val["abs_err"] = (y_val - val["pred_future_runs"]).abs()

val.groupby("balls_remaining")["abs_err"].mean().head(10)
val.groupby("balls_remaining")["abs_err"].mean().tail(10)


balls_remaining
111    25.612078
112    25.676110
113    25.511962
114    25.615991
115    25.775125
116    25.673514
117    25.983160
118    26.173778
119    26.210561
120    24.443346
Name: abs_err, dtype: float64

In [51]:
val.groupby("balls_remaining")["abs_err"].mean().head(10)


balls_remaining
1     3.620844
2     4.007031
3     4.512815
4     4.814528
5     5.187457
6     5.675943
7     5.987415
8     6.337052
9     6.599603
10    7.105896
Name: abs_err, dtype: float64

In [52]:
X_full = df_model[RPA_FEATURES].copy()


In [53]:
X_full_enc = cbe.transform(X_full)


In [54]:
X_full_enc.isna().sum().sum()


np.int64(369)

In [55]:
na_cols = X_full_enc.isna().sum()
na_cols[na_cols > 0].sort_values(ascending=False)


current_run_rate    369
dtype: int64

In [56]:
nan_idx = X_full_enc[X_full_enc.isna().any(axis=1)].index
len(nan_idx)


369

In [57]:
df_model.loc[nan_idx, [
    "match_id",
    "innings",
    "over",
    "ball",
    "balls_remaining",
    "current_run_rate"
]].head(10)


,match_id,innings,over,ball,balls_remaining,current_run_rate
11491,652307,1,0,1,120,NaN
1183,654059,1,0,1,120,NaN
10067,654091,1,0,1,120,NaN
13078,654097,1,0,1,120,NaN
13477,654099,1,0,1,120,NaN
17605,682935,1,0,1,120,NaN
43846,693093,1,0,1,120,NaN
44087,693095,1,0,1,120,NaN
44831,693101,1,0,1,120,NaN
49760,693147,1,0,1,120,NaN


In [58]:
df_model.loc[nan_idx, "innings"].value_counts()


innings
1    369
Name: count, dtype: int64

In [59]:
X_full_enc["current_run_rate"] = X_full_enc["current_run_rate"].fillna(0)


In [60]:
X_full_enc.isna().sum().sum()


np.int64(0)

In [61]:
df_model["pred_future_runs"] = rpa_model.predict(X_full_enc)


In [62]:
df_model["pred_future_runs"].describe()


count    707303.000000
mean         85.619073
std          46.235793
min         -28.288064
25%          47.173737
50%          85.297433
75%         123.575050
max         221.622265
Name: pred_future_runs, dtype: float64

In [64]:
df_model = df_model.sort_values(
    ["match_id", "innings", "over", "ball"]
).reset_index(drop=True)


In [65]:
df_model["future_runs_next"] = (
    df_model
    .groupby(["match_id", "innings"])["future_runs"]
    .shift(-1)
)

df_model["pred_future_runs_next"] = (
    df_model
    .groupby(["match_id", "innings"])["pred_future_runs"]
    .shift(-1)
)


In [66]:
df_model = df_model.dropna(
    subset=["future_runs_next", "pred_future_runs_next"]
)


In [67]:
df_model["rpa"] = (
    (df_model["future_runs"] - df_model["pred_future_runs"])
    -
    (df_model["future_runs_next"] - df_model["pred_future_runs_next"])
)


In [68]:
df_model.groupby(["match_id", "innings"])["rpa"].sum().describe()


count    5842.000000
mean        5.459450
std        27.914096
min       -98.585524
25%       -12.757907
50%         4.845654
75%        22.960480
max       122.986554
Name: rpa, dtype: float64

In [69]:
df_model["innings"].value_counts()


innings
1    701461
Name: count, dtype: int64

In [70]:
df_model["rpa"].shape


(701461,)

In [71]:
df_model[["match_id", "innings", "over", "ball", "rpa"]].head()


,match_id,innings,over,ball,rpa
0,636164,1,0,1,0.713676
1,636164,1,0,2,-0.565656
2,636164,1,0,3,-0.372242
3,636164,1,0,4,-0.679444
4,636164,1,0,5,-0.918300


In [72]:
df_model["rpa"].describe()


count    701461.000000
mean          0.045468
std           2.262093
min         -29.308000
25%          -0.904564
50%          -0.193533
75%           0.772129
max          32.586293
Name: rpa, dtype: float64

In [74]:
# =========================
# SAVE BALL-BY-BALL RPA MASTER
# =========================

BALL_BY_BALL_OUT = "outputs/ball_by_ball_rpa_master.csv"

BALL_COLS = [
    "match_id", "date", "league",
    "innings", "over", "ball",
    "batting_team", "bowling_team",
    "batter", "bowler",
    "total_runs", "is_wicket",
    "balls_remaining", "wickets_lost",
    "future_runs", "pred_future_runs",
    "rpa"
]

df_model[BALL_COLS].to_csv(BALL_BY_BALL_OUT, index=False)


In [75]:
# =========================
# BATTER RPA (1st innings)
# =========================

bat_rpa = (
    df_model
    .groupby("batter", as_index=False)
    .agg(
        balls_faced=("ball", "count"),
        total_rpa=("rpa", "sum"),
        avg_rpa_per_ball=("rpa", "mean")
    )
)

bat_rpa["rpa_per_100"] = bat_rpa["avg_rpa_per_ball"] * 100

# =========================
# BOWLER RPA (1st innings)
# =========================

bowl_rpa = (
    df_model
    .groupby("bowler", as_index=False)
    .agg(
        balls_bowled=("ball", "count"),
        total_rpa=("rpa", "sum"),
        avg_rpa_per_ball=("rpa", "mean")
    )
)

bowl_rpa["rpa_saved"] = -bowl_rpa["total_rpa"]
bowl_rpa["rpa_saved_per_100"] = -bowl_rpa["avg_rpa_per_ball"] * 100

# =========================
# SAVE CONSOLIDATED FILES
# =========================

bat_rpa.to_csv("outputs/batting_rpa_first_innings.csv", index=False)
bowl_rpa.to_csv("outputs/bowling_rpa_first_innings.csv", index=False)


In [76]:
def rpa_leaderboard(
    df,
    role="batting",          # "batting" or "bowling"
    league="all",            # league name or "all"
    start_date=None,         # "YYYY-MM-DD" or None
    min_balls=100,
    top_n=20
):
    """
    Generate RPA leaderboards with filtering.
    """

    data = df.copy()

    # -------------------------
    # FILTERS
    # -------------------------
    if league != "all":
        data = data[data["league"] == league]

    if start_date is not None:
        data = data[data["date"] >= pd.to_datetime(start_date)]

    # -------------------------
    # BATTING LEADERBOARD
    # -------------------------
    if role == "batting":
        out = (
            data
            .groupby("batter", as_index=False)
            .agg(
                balls=("ball", "count"),
                total_rpa=("rpa", "sum"),
                rpa_per_ball=("rpa", "mean")
            )
        )

        out["rpa_per_100"] = out["rpa_per_ball"] * 100
        out = out[out["balls"] >= min_balls]

        return out.sort_values(
            "total_rpa", ascending=False
        ).head(top_n)

    # -------------------------
    # BOWLING LEADERBOARD
    # -------------------------
    if role == "bowling":
        out = (
            data
            .groupby("bowler", as_index=False)
            .agg(
                balls=("ball", "count"),
                total_rpa=("rpa", "sum"),
                rpa_per_ball=("rpa", "mean")
            )
        )

        out["rpa_saved"] = -out["total_rpa"]
        out["rpa_saved_per_100"] = -out["rpa_per_ball"] * 100
        out = out[out["balls"] >= min_balls]

        return out.sort_values(
            "rpa_saved", ascending=False
        ).head(top_n)

    raise ValueError("role must be 'batting' or 'bowling'")


In [81]:
rpa_leaderboard(
    df_model,
    role="bowling",
    league="IPL",
    start_date="2025-01-01",
    min_balls=50,
    top_n=15
)

,bowler,balls,total_rpa,rpa_per_ball,rpa_saved,rpa_saved_per_100
38,Kuldeep Yadav,189,-43.430505,-0.229791,43.430505,22.979103
63,Noor Ahmad,187,-40.202874,-0.214989,40.202874,21.498863
30,JJ Bumrah,96,-24.641308,-0.256680,24.641308,25.668030
31,JR Hazlewood,105,-18.894235,-0.179945,18.894235,17.994509
14,CV Varun,142,-16.088034,-0.113296,16.088034,11.329601
49,MM Ali,67,-12.464054,-0.186031,12.464054,18.603065
17,E Malinga,50,-11.046875,-0.220938,11.046875,22.093751
96,YS Chahal,128,-9.419154,-0.073587,9.419154,7.358714
80,SN Thakur,51,-4.291587,-0.084149,4.291587,8.414876
21,HH Pandya,147,-1.852087,-0.012599,1.852087,1.259923


In [83]:
bowl_rpa.sort_values("rpa_saved_per_100", ascending=False).head(15)


,bowler,balls_bowled,total_rpa,avg_rpa_per_ball,rpa_saved,rpa_saved_per_100
915,JM James,17,-22.176122,-1.304478,22.176122,130.447776
1491,Niyas Nizar,24,-26.153902,-1.089746,26.153902,108.974593
204,Ali Shafiq,24,-25.826330,-1.076097,25.826330,107.609708
2186,V Kohli,11,-11.624843,-1.056804,11.624843,105.680393
78,AK Chahal,18,-18.778560,-1.043253,18.778560,104.325333
1311,Maaz Sadaqat,19,-19.747289,-1.039331,19.747289,103.933098
998,K Tripathi,24,-23.476687,-0.978195,23.476687,97.819529
1009,KD Pandey,12,-10.920496,-0.910041,10.920496,91.004130
1468,Nauman Ali,13,-11.631310,-0.894716,11.631310,89.471612
1041,KR Smuts,6,-5.273384,-0.878897,5.273384,87.889728
